### Section 0.1: Mounting Google Colab

In [ ]:
# Mount google drive
# Before start: you need to manually upload your downloaded nls_assignment.tar.gz to /content/drive/My Drive

from google.colab import drive
import os

# this command uses your own google drive as storage
# your google drive content will be in /content/drive/My Drive/ by default
drive.mount('/content/drive', force_remount=True)

my_dir = "/content/drive/My Drive"


# this command is equivalent to "cd" that sets a working directory
os.chdir(my_dir)

# extract files
!tar -xzf nls_assignment.tar.gz

### Section 0.1: Pani Raw Data
Retrieve a copy of your raw Pani scan at https://drive.google.com/drive/folders/1cAnk6iUSHb3oW5b0tTgxSJai--V3pfjC?usp=sharing, and place it under nls_assignment/data

For example, the desired folder structure should look something like:

```
nls_assignment/  
│── data/  
│   ├── processed_2025_03_06_15_18_03-Erich1/  
│   │   ├── preview.mp4  
│   │   ├── frame_bundle.npz
```

Be sure to check if your scan turned out OK; if it didn't, feel free to use one of the TA's scans instead for this assignment.




In [ ]:
# Check if Pani files are present

scan_folder = 'processed_2025_03_06_15_45_13-temp4'  # TODO: replace with your own folder name!
frame_bundle_path = os.path.join(my_dir, 'nls_assignment', 'data', scan_folder, 'frame_bundle.npz')

assert os.path.isfile(frame_bundle_path), f"File not found: {frame_bundle_path}"


### Section 0.2: Setup
If running this in Google Colab, make sure that you are connected to a GPU instance and run the install script below. It should (hopefully) take about 2-5mins to execute.

In [ ]:
import subprocess
import os
# Check if GPU exists

try:
    subprocess.check_output('nvidia-smi')
    print("GPU is enabled.")
    # Check if running in Google Colab
    if 'COLAB_GPU' in os.environ:
      # Instal TinyCuda
      %cd /content/

      !pip install -q gdown
      # cursed one-line wheel download/install
      !gdown "1Qx3Hn5Fpg_d5yjTyguCQ9wuaa-zBAH7n" -O tinycudann-colab-gpu.zip && unzip -o tinycudann-colab-gpu.zip && WHEEL=$(find . -maxdepth 1 -name "*.whl" | head -n 1) && echo "Found wheel: $WHEEL" && pip install "$WHEEL" --force-reinstall
      !pip install commentjson
      !pip install pytorch_lightning
      !pip install matplotlib==3.8.0
      # broken cuda version
      !pip uninstall -y torchaudio
    else:
      print("COLAB_GPU not detected")
except FileNotFoundError as e:
    print("GPU is not enabled in this notebook.")
    print("Please select 'Runtime -> Change runtime type' and set the hardware accelerator to GPU.")

## WARNING:
### Colab will ask to restart the session after running the above cell (because it pre-loads matplotlib for some reason). You should first restart the session, then continue running the cells below. Do not re-run the cell above after restarting the session.

In [ ]:
# Re-navigate to the correct folder location
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)


my_dir = "/content/drive/My Drive"
os.chdir(os.path.join(my_dir, 'nls_assignment'))
scan_folder = 'processed_2025_03_06_15_45_13-temp4'  # TODO: replace with your own folder name!
SCAN_FOLDER='processed_2025_03_06_15_45_13-temp4'  # TODO: replace with your own folder name!

### Section 1: Fitting a Neural Light Sphere

This section will cover how to fit our model to an input `frame_bundle.npz`, monitor the model's training, and plot its outputs.

First, we'll launch a tensorboard instance to montitor the model as it trains. `Note:` if you aren't running this in Google Colab you can alternately start a tensorboard instance outside this notebook pointing at `--logdir lightning_logs`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

To run `train.py` we just need to supply it with two required arguments, `--data_path` pointing to the location of the `frame_bundle.npz` and a name `--name` for the run:

In [ ]:
# TODO: replace with your own folder name!
!python3 train.py --data_path data/$SCAN_FOLDER/frame_bundle.npz --name $SCAN_FOLDER --num_batches 50 --max_epochs 50 --point_batch_size 25600

We also set `--max_epochs` to 50 and reduce `--point_batch_size` (the number of rays sampled per batch) to speed up training on the Colab instance at the cost of some reconstruction artifacts. You can remove these arguments to train the model for longer. Training might still take a couple minutes though, so feel free to grab a coffee at this point, or watch the training progress in tensorboard.

Once training is complete, we can load the model and use its `generate_outputs` function to visualize its outputs. This function takes as arguments: `height` and `width` in pixels for the desired output size; `u_lims` and `v_lims` to define the crop of the image (e.g., [0.1,0.9] would be an 80% center crop); `time` as a variable between [0,1] where 0 corresponds to the first captured frame, and 1 corresponds to the last frame; and `fov_scale` which scales the horizontal field of view of the render to simulate a wider angle camera.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import utils.utils as utils
from train import *

scan_folder = "processed_2025_03_06_15_45_13-temp4"
# Load model and data
model = PanoModel.load_from_checkpoint(f"checkpoints/{scan_folder}/last.ckpt", device="cuda", cached_data=f"checkpoints/{scan_folder}/data.pkl")
model = model.to("cuda")
model = model.eval()
model.load_volume()

# Generate outputs (reference image, rendered image)
rgb_reference, rgb, _, _ = model.generate_outputs(height=1080, width=1440, u_lims=[0, 1], v_lims=[0, 1], time=0.55, fov_scale=2.4)
brightness = 1.1

# Apply brightness for visualization
rgb = (rgb * brightness).clamp(0,1)
rgb_reference = (rgb_reference * brightness).clamp(0,1)

# Plot images
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(rgb_reference.permute(1, 2, 0).cpu())
ax[0].set_aspect(1.6)
ax[0].set_title("Reference")
ax[1].imshow(rgb.permute(1, 2, 0).cpu() )
ax[1].set_title("Wide FOV Re-Render")

for a in ax:
    a.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

plt.subplots_adjust(wspace=0.0)
# plt.show()
plt.savefig('visualization.png')


/home/yhu/miniconda3/envs/nls/lib/python3.10/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce RTX 5090 which is of cuda capability 12.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (5.0) - (9.0)
    
  warnings.warn(
/home/yhu/miniconda3/envs/nls/lib/python3.10/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.8 12.9 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
/home/yhu/miniconda3/envs/nls/lib/python3.10/site-packages/torch/cuda/__init__.py:326: UserWarning: 
NVIDIA GeForce RTX 5090 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_89 sm_90 compute_90.
If you want to use the NVIDIA GeForce RTX 5090 GPU with PyTo

Nice, now we can fully appreciate how we can turn an already ultrawide capture into a ~140 degree fisheye render.

**Advanced:** The code below is essentially the render preview tool from `render.py` rewritten as a Jupyter widget. Rather than call `generate_outputs` to generate images, it overviews how to generate rays and manually sample the neural light sphere model via its `inference(t, uv, ray_origins, ray_directions)` function.

Annotated in the comments, the steps we take to generate an image is:
1. Query `model.model_rotation` and `model.model_translation` to get the camera's estimated position and rotation at an input time `t` [0-1] (learned during training).
2. Offset this position and rotation by the user's input.
3. Retrieve the inverse camera intrinsics matrix for time `t` (recorded by the phone), scaled by user input (to widen/narrow FOV).
4. Generate grid of image coordinates (u,v) with `utils.make_grid`, scaled by user input (to widen/narrow FOV).
5. From the rotation, inverse intrinsics, and image coordinate, generate ray directions via pinhole projection with `model.generate_ray_directions`. Apply lens distortion unless `model.args.no_lens_distortion` is `True`.
6. Sample the neural light sphere model via these ray origins, directions, and image coordinates to produce output colors, scaled by `brightness` before reshaped into an image array.


In [3]:
from IPython.display import display, clear_output
import ipywidgets as widgets
from PIL import Image
from io import BytesIO

# Load model
model = PanoModel.load_from_checkpoint(f"checkpoints/{scan_folder}/last.ckpt", device="cuda", cached_data=f"checkpoints/{scan_folder}/data.pkl")
model = model.to("cuda")
model = model.eval()

# Rendered image size
width = 960
height = 640

# Define default values for widgets
default_values = {  'time': 0.55, 'brightness': 1.1, 'fov_scale': 1.0,
    'offset_x': 0.0, 'offset_y': 0.0, 'offset_z': 0.0,
    'offset_qw': 0.0, 'offset_qx': 0.0, 'offset_qy': 0.0, 'offset_qz': 0.0,
    'ray_offset': True, 'view_color': True, 'lens_distortion': True
}

# Create an Image widget for displaying the rendered image
img_widget = widgets.Image(format='png')

# Render script
def render_image(time=default_values['time'], brightness=default_values['brightness'], fov_scale=default_values['fov_scale'],
                offset_x=default_values['offset_x'], offset_y=default_values['offset_y'], offset_z=default_values['offset_z'],
                offset_qw=default_values['offset_qw'], offset_qx=default_values['offset_qx'], offset_qy=default_values['offset_qy'],
                offset_qz=default_values['offset_qz'], ray_offset=default_values['ray_offset'], view_color=default_values['view_color'],
                lens_distortion=default_values['lens_distortion']):  # Added lens_distortion parameter

    model.args.no_offset = not ray_offset  # if no_offset, zero out ray offset model
    model.args.no_view_color = not view_color  # if no_view_color, zero out view dependent color model
    model.args.no_lens_distortion = not lens_distortion  # if no_lens_distortion, zero out lens distortion model

    # Offset to camera center
    offset_translation = torch.tensor([offset_x, offset_y, offset_z], device="cuda", dtype=torch.float32)
    # Offset to camera rotation (as wxyz quaternion)
    offset_quaternion = torch.tensor([offset_qw, offset_qx, offset_qy, offset_qz], device="cuda", dtype=torch.float32)

    # Time [0-1], used to sample camera position/rotation from recorded data
    t_tensor = torch.full((width * height,), time, device="cuda", dtype=torch.float32) # B,

    frame_index = int(time * model.args.num_frames - 1)
    frame_index = max(0, min(frame_index, model.args.num_frames - 1))  # frame (integer) corresponding to this time

    intrinsics_inv = model.data.intrinsics_inv[frame_index].clone()  # camera intrinsics (inverse)
    intrinsics_inv[0] *= fov_scale  # multiply fx, cx by fov_scale to stretch horizontally
    intrinsics_inv = intrinsics_inv.unsqueeze(0).repeat(width * height, 1, 1).to("cuda")  # B, 3, 3

    quaternion_camera_to_world = model.data.quaternion_camera_to_world[frame_index].to("cuda")  # camera rotation (gyroscope)
    quaternion_camera_to_world += offset_quaternion  # offset
    quaternion_camera_to_world /= quaternion_camera_to_world.norm(dim=-1, keepdim=True)  # renormalize

    camera_to_world = model.model_rotation(quaternion_camera_to_world, t_tensor).to("cuda")  # learned rotation offset
    ray_origins = model.model_translation(t_tensor, 1.0) + offset_translation  # learned translation

    uv = utils.make_grid(height, width, [0, 1], [0, 1]).to("cuda")  # u,v image coordinates
    adjusted_uv = uv * fov_scale + 0.5 * (1 - fov_scale)  # re-scale coordinate to match FOV

    # Camera projection to get world space ray directions
    ray_directions = model.generate_ray_directions(adjusted_uv, camera_to_world, intrinsics_inv)
    with torch.no_grad():
        # Neural light sphere model forward pass, clamp (u,v) to not violate bounds of neural field encoding
        rgb_transmission = model.inference(t_tensor, adjusted_uv.clamp(0,1), ray_origins, ray_directions, 1.0)

    # Convert to byte image
    rgb_image = model.color_and_tone(rgb_transmission, height, width).permute(1, 2, 0).detach().cpu()
    rgb_image = (rgb_image * brightness).clamp(0, 1).numpy()
    pil_image = Image.fromarray((rgb_image * 255).astype(np.uint8))
    buf = BytesIO()
    pil_image.save(buf, format='PNG')
    img_widget.value = buf.getvalue()


The script below runs this in an interactive jupyter widget, with sliders to offset the rendered camera pose and FOV, and switches to turn model components off and on.

**Note:** In Google Colab this can be quite laggy, wait a second or two after changing a switch/slider to let it update the image.

In [4]:

display(img_widget)

# Define sliders and switches
time_slider = widgets.FloatSlider(value=default_values['time'], min=0.0, max=1.0, step=0.01, description='Time:', continuous_update=False)
brightness_slider = widgets.FloatSlider(value=default_values['brightness'], min=0.0, max=2.0, step=0.1, description='Brightness:', continuous_update=False)
fov_scale_slider = widgets.FloatSlider(value=default_values['fov_scale'], min=0.5, max=2.0, step=0.1, description='FOV Scale:', continuous_update=False)
offset_x_slider = widgets.FloatSlider(value=default_values['offset_x'], min=-1.0, max=1.0, step=0.1, description='Offset X:', continuous_update=False)
offset_y_slider = widgets.FloatSlider(value=default_values['offset_y'], min=-1.0, max=1.0, step=0.1, description='Offset Y:', continuous_update=False)
offset_z_slider = widgets.FloatSlider(value=default_values['offset_z'], min=-1.0, max=1.0, step=0.1, description='Offset Z:', continuous_update=False)
offset_qw_slider = widgets.FloatSlider(value=default_values['offset_qw'], min=-1.0, max=1.0, step=0.1, description='offset_qw:', continuous_update=False)
offset_qx_slider = widgets.FloatSlider(value=default_values['offset_qx'], min=-1.0, max=1.0, step=0.1, description='offset_qx:', continuous_update=False)
offset_qy_slider = widgets.FloatSlider(value=default_values['offset_qy'], min=-1.0, max=1.0, step=0.1, description='offset_qy:', continuous_update=False)
offset_qz_slider = widgets.FloatSlider(value=default_values['offset_qz'], min=-1.0, max=1.0, step=0.1, description='offset_qz:', continuous_update=False)
ray_offset_switch = widgets.Checkbox(value=default_values['ray_offset'], description='ray_offset:', continuous_update=False)
view_color_switch = widgets.Checkbox(value=default_values['view_color'], description='view_color:', continuous_update=False)
lens_distortion_switch = widgets.Checkbox(value=default_values['lens_distortion'], description='Lens Distortion:', continuous_update=False)
reset_button = widgets.Button(description="Reset", button_style='')

# Define reset functionality
def on_reset_button_clicked(b):
    for widget, default in zip(
        [time_slider, brightness_slider, fov_scale_slider, offset_x_slider, offset_y_slider, offset_z_slider,
         offset_qw_slider, offset_qx_slider, offset_qy_slider, offset_qz_slider, ray_offset_switch, view_color_switch,
         lens_distortion_switch],  # Include lens_distortion_switch
        [default_values['time'], default_values['brightness'], default_values['fov_scale'],
         default_values['offset_x'], default_values['offset_y'], default_values['offset_z'],
         default_values['offset_qw'], default_values['offset_qx'], default_values['offset_qy'],
         default_values['offset_qz'], default_values['ray_offset'], default_values['view_color'],
         default_values['lens_distortion']]
    ):
        widget.value = default

reset_button.on_click(on_reset_button_clicked)

# Organize widgets into layout
slider_box = widgets.VBox([
    widgets.HBox([time_slider, brightness_slider, fov_scale_slider]),
    widgets.HBox([offset_x_slider, offset_y_slider, offset_z_slider]),
    widgets.HBox([offset_qw_slider, offset_qx_slider, offset_qy_slider, offset_qz_slider]),
    widgets.HBox([ray_offset_switch, view_color_switch, lens_distortion_switch]),
    widgets.HBox([reset_button])
])

display(slider_box)

# Link widgets to the rendering function
out = widgets.interactive_output(
    render_image,
    {
        'time': time_slider, 'brightness': brightness_slider, 'fov_scale': fov_scale_slider,
        'offset_x': offset_x_slider, 'offset_y': offset_y_slider, 'offset_z': offset_z_slider,
        'offset_qw': offset_qw_slider, 'offset_qx': offset_qx_slider, 'offset_qy': offset_qy_slider, 'offset_qz': offset_qz_slider,
        'ray_offset': ray_offset_switch, 'view_color': view_color_switch, 'lens_distortion': lens_distortion_switch
    }
)

display(out)

Image(value=b'')

Output()